# Answering EDA (read-only)

This notebook inspects persisted Feature 10 answer traces and their immutable evidence packets. It never writes SQLite, indexes, `Courses/`, prompts, API keys, or model output. Run the normal pipeline first (for example `uv run -m uni_rag_agent ask ...`). Empty tables are expected on a fresh database.

In [ ]:
from pathlib import Path
import json
import sqlite3
import pandas as pd
from uni_rag_agent.answering.audit import audit_stored_answer

root = Path.cwd().resolve()
if root.name.lower() == "notebooks":
    root = root.parent
sqlite_path = root / "data" / "uni_rag.sqlite"
uri = sqlite_path.as_uri() + "?mode=ro"
connection = sqlite3.connect(uri, uri=True)
connection.execute("PRAGMA query_only = ON")
assert connection.execute("PRAGMA query_only").fetchone()[0] == 1

In [ ]:
def read_json(value):
    if not isinstance(value, str) or not value.strip():
        return None, False, "missing or blank JSON"
    try:
        parsed = json.loads(value)
    except (TypeError, json.JSONDecodeError):
        return None, False, "malformed JSON"
    return parsed, True, "valid"


answers = pd.read_sql_query(
    """SELECT answers.id AS answer_id, answers.evidence_packet_id,
              answers.answer_text, answers.citations_json,
              answers.limitations_json, answers.model_name, answers.created_at,
              evidence_packets.search_run_id, evidence_packets.evidence_count
       FROM answers JOIN evidence_packets
         ON evidence_packets.id = answers.evidence_packet_id
       ORDER BY answers.id""",
    connection,
)
answers["limitations_parse"] = answers["limitations_json"].map(read_json)
answers["limitations"] = answers["limitations_parse"].map(
    lambda result: result[0] if result[1] and isinstance(result[0], list) else []
)
answers["limitations_parsed"] = answers["limitations_parse"].map(
    lambda result: result[1] and isinstance(result[0], list)
)
answers["limitation_count"] = answers["limitations"].map(len)
answers.head()

In [ ]:
packets = pd.read_sql_query(
    """SELECT id AS evidence_packet_id, search_run_id, evidence_count, packet_json, created_at
       FROM evidence_packets ORDER BY id""",
    connection,
)
packets["packet_parse"] = packets["packet_json"].map(read_json)
packets["packet"] = packets["packet_parse"].map(
    lambda result: result[0] if result[1] and isinstance(result[0], dict) else {}
)
packets["packet_parsed"] = packets["packet_parse"].map(
    lambda result: result[1] and isinstance(result[0], dict)
)
packets["weakness_count"] = packets["packet"].map(
    lambda value: len(value.get("weaknesses", []))
)
packets[
    ["evidence_packet_id", "search_run_id", "evidence_count", "weakness_count"]
].head()

## Checks

Citation validity preserves JSON parse status, requires canonical `E<n>` ids, compares every structured citation field to the immutable packet evidence at `evidence_index`, and verifies rendered markers/references. Runtime `chunk:<id>` aliases are canonicalized before storage and therefore never appear in `citations_json`. Legitimate empty citation arrays remain valid for deterministic insufficient-evidence and safe-refusal rows. Limitations are expected for weak packets. Provider/model traces are stored only as `provider:model`; prompts, keys, and invalid raw responses are intentionally absent. Injected test chat models are used by pytest and are not invoked by this notebook.

In [ ]:
if not answers.empty:
    joined = answers.merge(
        packets[["evidence_packet_id", "packet_json"]],
        on="evidence_packet_id",
        how="left",
    )
    joined["citation_audit"] = joined.apply(
        lambda row: audit_stored_answer(
            row["citations_json"], row["packet_json"], row["answer_text"]
        ),
        axis=1,
    )
    joined["citations"] = joined["citation_audit"].map(lambda value: value["citations"])
    joined["citation_count"] = joined["citations"].map(len)
    joined["citations_valid"] = joined["citation_audit"].map(
        lambda value: value["valid"]
    )
    joined["citation_diagnostic"] = joined["citation_audit"].map(
        lambda value: value["diagnostic"]
    )
    joined["citations_json_parsed"] = joined["citation_audit"].map(
        lambda value: value["citations_parsed"]
    )
    joined[
        [
            "answer_id",
            "evidence_packet_id",
            "citation_count",
            "limitation_count",
            "citations_valid",
            "citation_diagnostic",
            "citations_json_parsed",
            "model_name",
        ]
    ]
else:
    joined = pd.DataFrame(
        columns=["answer_id", "evidence_packet_id", "citations_valid"]
    )
joined

In [ ]:
# Close the read-only handle when exploration is complete; no commit/write is ever issued.
connection.close()